<a href="https://colab.research.google.com/github/CarlosNoriegaPolo/fMetRecogninPrediction/blob/colab_development/fMet_Recognin_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **fMet Recognin Prediction Project**

### **Coding Rules for This Project**

*   **Write Clear Code**: Use meaningful variable and function names.

*   **Comment Your Code**: Briefly explain your code and reasoning.

*   **Keep It Organized**: Use different sections to develop and test individual portions of the code.

*   **Incorporate Checks**: Frequently check the output of code blocks by printing statements and execution times

# Part 0. Load libraries

Load all the neccesary libraries here.

In [28]:
import requests # for API requests
import re # regular expressions
import time # to control running times
from tqdm import tqdm # progress bars
import pandas as pd # to make DataFrame

# Part 1. Retrieve Protein Models - Data Mining

## Get all reviewed human proteins from UniProt API

In [ ]:
# Set parameters
batch_size = 500 # it will retrieve entries in groups of 500 (can change)
base_url = "https://rest.uniprot.org/uniprotkb/search"
re_next_link = re.compile(r'<(.+)>; rel="next"')

# Set up session with retry strategy
session = requests.Session()
retry_strategy = requests.adapters.Retry(
    total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504] # these are the codes for failed attempts
)
session.mount("https://", requests.adapters.HTTPAdapter(max_retries=retry_strategy))

# Construct query parameters (explicitly requesting only the necessary fields)
params = {
    'query': '(reviewed:true) AND (organism_id:9606)',  # 9606 is the taxonomy ID for humans
    'format': 'tsv',
    'fields': 'accession,protein_name',  # Request only UniProt ID and protein name
    'size': batch_size
}

# Construct initial URL
url = f"{base_url}?{'&'.join(f'{k}={v}' for k, v in params.items())}"

# Initialize dictionary to store protein data
uniprot_IDs = []
protein_IDs = {}
progress = 0

# Start timer for total time
start_time = time.time()

# Fetch and process batches
while url:
    try:
        # Send request and get response
        response = session.get(url)
        response.raise_for_status()

        # Parse the response
        lines = response.text.splitlines()

        # Process the data (skip header)
        for line in lines[1:]:  # Skip the header
            fields = line.split("\t")
            if len(fields) >= 2:  # Ensure there are at least two columns
                uniprot_id, protein_name = fields[:2]  # Take only the first two values

                uniprot_IDs.append(uniprot_id)
                protein_IDs[protein_name] = uniprot_id

        # Update progress
        progress += len(lines[1:])
        total = response.headers.get("x-total-results", "unknown")
        print(f"Progress: {progress:,} / {int(total):,} entries",
              f"({progress/int(total)*100:.1f}%)")

        # Extract the next page URL from headers (pagination)
        next_link = response.headers.get("Link")
        if next_link:
            match = re.search(r'<(.+)>; rel="next"', next_link)
            url = match.group(1) if match else None
        else:
            url = None

    except requests.exceptions.RequestException as e:
        print(f"Error fetching batch: {e}")
        break

# Calculate and print total time
duration = time.time() - start_time
print(f"\nDownload completed in {duration:.1f} seconds")
print(f"Total proteins: {len(protein_IDs)}")

Progress: 500 / 20,417 entries (2.4%)
Progress: 1,000 / 20,417 entries (4.9%)
Progress: 1,500 / 20,417 entries (7.3%)
Progress: 2,000 / 20,417 entries (9.8%)
Progress: 2,500 / 20,417 entries (12.2%)
Progress: 3,000 / 20,417 entries (14.7%)
Progress: 3,500 / 20,417 entries (17.1%)
Progress: 4,000 / 20,417 entries (19.6%)
Progress: 4,500 / 20,417 entries (22.0%)
Progress: 5,000 / 20,417 entries (24.5%)
Progress: 5,500 / 20,417 entries (26.9%)
Progress: 6,000 / 20,417 entries (29.4%)
Progress: 6,500 / 20,417 entries (31.8%)
Progress: 7,000 / 20,417 entries (34.3%)
Progress: 7,500 / 20,417 entries (36.7%)
Progress: 8,000 / 20,417 entries (39.2%)
Progress: 8,500 / 20,417 entries (41.6%)
Progress: 9,000 / 20,417 entries (44.1%)
Progress: 9,500 / 20,417 entries (46.5%)
Progress: 10,000 / 20,417 entries (49.0%)
Progress: 10,500 / 20,417 entries (51.4%)
Progress: 11,000 / 20,417 entries (53.9%)
Progress: 11,500 / 20,417 entries (56.3%)
Progress: 12,000 / 20,417 entries (58.8%)
Progress: 12,500 

In [ ]:
# Check the first 10 items of the protein_IDs dictionary
for i, (protein_name, uniprot_id) in enumerate(protein_IDs.items()):
    if i < 10:
        print(f"Protein ID: {uniprot_id} Protein Name: {protein_name}")

Protein ID: A0A0C5B5G6 Protein Name: Mitochondrial-derived peptide MOTS-c (Mitochondrial open reading frame of the 12S rRNA-c)
Protein ID: A0A1B0GTW7 Protein Name: Ciliated left-right organizer metallopeptidase (EC 3.4.24.-) (Leishmanolysin-like peptidase 2)
Protein ID: A0JNW5 Protein Name: Bridge-like lipid transfer protein family member 3B (Syntaxin-6 Habc-interacting protein of 164 kDa) (UHRF1-binding protein 1-like)
Protein ID: A0JP26 Protein Name: POTE ankyrin domain family member B3
Protein ID: A0PK11 Protein Name: Clarin-2
Protein ID: A1A4S6 Protein Name: Rho GTPase-activating protein 10 (GTPase regulator associated with focal adhesion kinase 2) (GRAF2) (Graf-related protein 2) (Rho-type GTPase-activating protein 10)
Protein ID: A1A519 Protein Name: Protein FAM170A (Zinc finger domain-containing protein) (Zinc finger protein ZNFD)
Protein ID: A1L190 Protein Name: Synaptonemal complex central element protein 3 (Testis highly expressed gene 2 protein) (THEG-2)
Protein ID: A1L3X0 P

## Get data for **features** and explore it

Start by using the UniProt IDs stored in the protein_IDs dictionary. Using the UniProt API, extract all information regarding the features for each protein and append this information into a dataframe. Then analyse this further to completely understand our data i.e. what are all of the possible labels, what are the average positions/lengths of each feature, etc.

In [81]:
# Use the previously generated protein_IDs dictionary to create a list of all UniProt IDs to fetch
uniprot_ids_to_fetch = list(protein_IDs.values())

# Set parameters
batch_size = 200  # Slightly smaller batch size when we want to search for more information
base_url = "https://rest.uniprot.org/uniprotkb/search"

# Set up session with retry strategy
session = requests.Session()
retry_strategy = requests.adapters.Retry(
    total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504]
)
session.mount("https://", requests.adapters.HTTPAdapter(max_retries=retry_strategy))

# Prepare an empty DataFrame to store results
features_df = pd.DataFrame(columns=['accession', 'protein_name', 'total_length',
                                   'signal_peptide', 'signal_start', 'signal_end',
                                   'transit_peptide', 'transit_start', 'transit_end'])

# Process in batches to avoid overwhelming the API
total_ids = len(uniprot_ids_to_fetch)
processed = 0
batch_num = 0

# Start timer
start_time = time.time()

# Process in batches
for i in range(0, total_ids, batch_size):
    batch_num += 1
    batch_ids = uniprot_ids_to_fetch[i:i+batch_size]

    # Construct the query with OR conditions for each ID
    id_query = " OR ".join([f"accession:{acc}" for acc in batch_ids])

    # Construct query parameters
    params = {
        'query': f'({id_query})',
        'format': 'tsv',
        'fields': 'accession,protein_name,ft_signal,ft_transit,length',
        'size': batch_size
    }

    # Construct initial URL
    url = f"{base_url}?{'&'.join(f'{k}={v}' for k, v in params.items())}"

    try:
        # Send request and get response
        response = session.get(url)
        response.raise_for_status()

        # Parse the response
        lines = response.text.splitlines()

        # Process the data (skip header)
        batch_data = []
        for line in lines[1:]:  # Skip the header
            fields = line.split("\t")

            # Ensure we have at least 5 columns (some may be empty)
            if len(fields) < 5:
                fields.extend([''] * (5 - len(fields)))

            # Extract basic information
            accession = fields[0]
            protein_name = fields[1]
            ft_signal = fields[2] if fields[2] else 'None'
            ft_transit = fields[3] if fields[3] else 'None'
            total_length = fields[4]

            # Process signal peptide information
            signal_peptide = 0
            signal_start = None
            signal_end = None

            if ft_signal != 'None':
                signal_peptide = 1
                signal_match = re.search(r'SIGNAL (\d+)\.\.(\d+)', ft_signal)
                if signal_match:
                    signal_start = int(signal_match.group(1))
                    signal_end = int(signal_match.group(2))

            # Process transit peptide information
            transit_peptide = 0
            transit_start = None
            transit_end = None

            if ft_transit != 'None':
                transit_peptide = 1
                transit_match = re.search(r'TRANSIT (\d+)\.\.(\d+)', ft_transit)
                if transit_match:
                    transit_start = int(transit_match.group(1))
                    transit_end = int(transit_match.group(2))

            # Add processed data to the batch
            batch_data.append([
                accession, protein_name, total_length,
                signal_peptide, signal_start, signal_end,
                transit_peptide, transit_start, transit_end
            ])

        # Add to DataFrame
        temp_df = pd.DataFrame(batch_data, columns=[
            'accession', 'protein_name', 'total_length',
            'signal_peptide', 'signal_start', 'signal_end',
            'transit_peptide', 'transit_start', 'transit_end'
        ])
        features_df = pd.concat([features_df, temp_df], ignore_index=True)

        # Update progress
        processed += len(batch_ids)
        print(f"Progress: {processed:,} / {total_ids:,} proteins ({processed/total_ids*100:.1f}%)")

        # Sleep briefly to avoid overwhelming the API
        time.sleep(0.5)

    except requests.exceptions.RequestException as e:
        print(f"Error fetching batch {batch_num}: {e}")
        # Continue with next batch instead of breaking
        continue

# Calculate and print total time
duration = time.time() - start_time
print(f"\nData extraction completed in {duration:.1f} seconds")
print(f"Total proteins processed: {processed:,} out of {total_ids:,}")

# Convert length to integer where possible
features_df['total_length'] = pd.to_numeric(features_df['total_length'], errors='coerce')

<ipython-input-81-2e0f62e63c0f>:108: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  features_df = pd.concat([features_df, temp_df], ignore_index=True)


Progress: 200 / 20,417 proteins (1.0%)
Progress: 400 / 20,417 proteins (2.0%)
Progress: 600 / 20,417 proteins (2.9%)
Progress: 800 / 20,417 proteins (3.9%)
Progress: 1,000 / 20,417 proteins (4.9%)
Progress: 1,200 / 20,417 proteins (5.9%)
Progress: 1,400 / 20,417 proteins (6.9%)
Progress: 1,600 / 20,417 proteins (7.8%)
Progress: 1,800 / 20,417 proteins (8.8%)
Progress: 2,000 / 20,417 proteins (9.8%)
Progress: 2,200 / 20,417 proteins (10.8%)
Progress: 2,400 / 20,417 proteins (11.8%)
Progress: 2,600 / 20,417 proteins (12.7%)
Progress: 2,800 / 20,417 proteins (13.7%)
Progress: 3,000 / 20,417 proteins (14.7%)
Progress: 3,200 / 20,417 proteins (15.7%)
Progress: 3,400 / 20,417 proteins (16.7%)
Progress: 3,600 / 20,417 proteins (17.6%)
Progress: 3,800 / 20,417 proteins (18.6%)
Progress: 4,000 / 20,417 proteins (19.6%)
Progress: 4,200 / 20,417 proteins (20.6%)
Progress: 4,400 / 20,417 proteins (21.6%)
Progress: 4,600 / 20,417 proteins (22.5%)
Progress: 4,800 / 20,417 proteins (23.5%)
Progress: 

KeyboardInterrupt: 

In [49]:
# Check the dataframe
features_df.head(50)

,accession,protein_name,total_length,signal_peptide,signal_start,signal_end,transit_peptide,transit_start,transit_end
0,A0A0C5B5G6,Mitochondrial-derived peptide MOTS-c (Mitochon...,16,0,NaN,NaN,0,NaN,NaN
1,A0A1B0GTW7,Ciliated left-right organizer metallopeptidase...,788,1,1.0,20.0,0,NaN,NaN
2,A0JNW5,Bridge-like lipid transfer protein family memb...,1464,0,NaN,NaN,0,NaN,NaN
3,A0JP26,POTE ankyrin domain family member B3,581,0,NaN,NaN,0,NaN,NaN
4,A0PK11,Clarin-2,232,0,NaN,NaN,0,NaN,NaN
5,A1A4S6,Rho GTPase-activating protein 10 (GTPase regul...,786,0,NaN,NaN,0,NaN,NaN
6,A1L190,Synaptonemal complex central element protein 3...,88,0,NaN,NaN,0,NaN,NaN
7,A1L3X0,Very long chain fatty acid elongase 7 (EC 2.3....,281,0,NaN,NaN,0,NaN,NaN
8,A1X283,SH3 and PX domain-containing protein 2B (Adapt...,911,0,NaN,NaN,0,NaN,NaN
9,A2A2Y4,FERM domain-containing protein 3 (Band 4.1-lik...,597,0,NaN,NaN,0,NaN,NaN


In [60]:
# Exploratory analysis

# Calculate total number of proteins with signal peptides
print(f"Proteins with signal peptides: {(features_df['signal_peptide'] == 1).sum()}")

# Calculate average length of all signal peptides
if features_df['signal_peptide'].sum() > 0:
    signal_lengths = features_df.loc[features_df['signal_peptide'] == 1, 'signal_end'] - \
                    features_df.loc[features_df['signal_peptide'] == 1, 'signal_start'] + 1
    print(f"Average signal peptide length: {signal_lengths.mean():.1f} amino acids")

# Check if all signal peptides start at amino acid 1
all_signal_start_ones = features_df['signal_start'].dropna().all()
print(f"All signal peptides start at amino acid 1: {all_signal_start_ones}")

# Calculate total number of proteins with transit peptides
print(f"Proteins with transit peptides: {(features_df['transit_peptide'] == 1).sum()}")

# Calculate average length of all transit peptides
if features_df['transit_peptide'].sum() > 0:
    transit_lengths = features_df.loc[features_df['transit_peptide'] == 1, 'transit_end'] - \
                     features_df.loc[features_df['transit_peptide'] == 1, 'transit_start'] + 1
    print(f"Average transit peptide length: {transit_lengths.mean():.1f} amino acids")

# Check if all transit peptides start at amino acid 1
all_transit_start_ones = features_df['transit_start'].dropna().all()
print(f"All transit peptides start at amino acid 1: {all_signal_start_ones}")

# Calculate total number of proteins with both signal and transit peptides
proteins_with_both = features_df[(features_df['signal_peptide'] == 1) & (features_df['transit_peptide'] == 1)]
print(f"Proteins with both signal and transit peptides: {len(proteins_with_both)}")

Proteins with signal peptides: 3611
Average signal peptide length: 24.1 amino acids
All signal peptides start at amino acid 1: True
Proteins with transit peptides: 562
Average transit peptide length: 37.6 amino acids
All transit peptides start at amino acid 1: True
Proteins with both signal and transit peptides: 0


## Get data for **models** and explore it

Start by using the UniProt IDs stored in the protein_IDs dictionary. Using the UniProt API, extract all information regarding the protein models available for each protein and append this information into a separate dataframe. Then analyse this further to completely understand our data i.e. how many proteins do not have any models, only Alphafold, etc.

### Get PDB informations
 #### Process each PDB_model into -> {'Source':model_source,'ID':model_id,'Method':model_method,'Resolution':model_resol,'Chains':model_chains}

In [ ]:
# Set parameters
batch_size = 500 # it will retrieve entries in groups of 500 (can change)
base_url = "https://rest.uniprot.org/uniprotkb/search"
re_next_link = re.compile(r'<(.+)>; rel="next"')

# Set up session with retry strategy
session = requests.Session()
retry_strategy = requests.adapters.Retry(
    total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504] # these are the codes for failed attempts
)
session.mount("https://", requests.adapters.HTTPAdapter(max_retries=retry_strategy))

# Construct query parameters (explicitly requesting only the necessary fields)
params = {
    'query': '(reviewed:true) AND (organism_id:9606)',  # 9606 is the taxonomy ID for humans
    'format': 'json',
    'fields': 'accession,structure_3d',
    'size': batch_size
}

# Construct initial URL
url = f"{base_url}?{'&'.join(f'{k}={v}' for k, v in params.items())}"

# Initialize dictionary to store protein data
uniprot_IDs = {}

models = {}
model_count = {}

progress = 0

# Start timer for total time
start_time = time.time()

# Fetch and process batches
while url:
    try:
        # Send request and get response
        response = session.get(url)
        response.raise_for_status()

        # Parse the response
        databases = response.json()['results'] #len(database) = batch_size

        # Process the data (skip header)
        for database in databases:
            uniprot_id = database['primaryAccession']
            model_DBs = database['uniProtKBCrossReferences']
            PDBs = []

            for DB in model_DBs: #each model in model_db:
                model_source, model_id, model_prop = DB['database'], DB['id'], DB['properties']

                model_method = [item for item in model_prop if item.get('key') == 'Method'][0]['value']
                model_resol = [item for item in model_prop if item.get('key') == 'Resolution'][0]['value']
                model_chains = [item for item in model_prop if item.get('key') == 'Chains'][0]['value']

                model_info = {'Source':model_source,
                              'ID':model_id,
                              'Method':model_method,
                              'Resolution':model_resol,
                              'Chains':model_chains}
                PDBs.append(model_info)

            uniprot_IDs[uniprot_id] = uniprot_id

            models[uniprot_id] = PDBs
            model_count[uniprot_id] = len(model_DBs)

        # Update progress
        progress += len(databases)
        total = response.headers.get("x-total-results", "unknown")
        print(f"Progress: {progress:,} / {int(total):,} entries",
              f"({progress/int(total)*100:.1f}%)")

        # Extract the next page URL from headers (pagination)
        next_link = response.headers.get("Link")
        if next_link:
            match = re.search(r'<(.+)>; rel="next"', next_link)
            url = match.group(1) if match else None
        else:
            url = None

    except requests.exceptions.RequestException as e:
        print(f"Error fetching batch: {e}")
        break

# Calculate and print total time
duration = time.time() - start_time
print(f"\nDownload completed in {duration:.1f} seconds")
print(f"Total entries: {len(uniprot_IDs)}")

Progress: 500 / 20,417 entries (2.4%)
Progress: 1,000 / 20,417 entries (4.9%)
Progress: 1,500 / 20,417 entries (7.3%)
Progress: 2,000 / 20,417 entries (9.8%)
Progress: 2,500 / 20,417 entries (12.2%)
Progress: 3,000 / 20,417 entries (14.7%)
Progress: 3,500 / 20,417 entries (17.1%)
Progress: 4,000 / 20,417 entries (19.6%)
Progress: 4,500 / 20,417 entries (22.0%)
Progress: 5,000 / 20,417 entries (24.5%)
Progress: 5,500 / 20,417 entries (26.9%)
Progress: 6,000 / 20,417 entries (29.4%)
Progress: 6,500 / 20,417 entries (31.8%)
Progress: 7,000 / 20,417 entries (34.3%)
Progress: 7,500 / 20,417 entries (36.7%)
Progress: 8,000 / 20,417 entries (39.2%)
Progress: 8,500 / 20,417 entries (41.6%)
Progress: 9,000 / 20,417 entries (44.1%)
Progress: 9,500 / 20,417 entries (46.5%)
Progress: 10,000 / 20,417 entries (49.0%)
Progress: 10,500 / 20,417 entries (51.4%)
Progress: 11,000 / 20,417 entries (53.9%)
Progress: 11,500 / 20,417 entries (56.3%)
Progress: 12,000 / 20,417 entries (58.8%)
Progress: 12,500 

#### Data

In [ ]:
models

{'A0A0C5B5G6': [],
 'A0A1B0GTW7': [],
 'A0JNW5': [],
 'A0JP26': [],
 'A0PK11': [],
 'A1A4S6': [{'Source': 'PDB',
   'ID': '2MIO',
   'Method': 'NMR',
   'Resolution': '-',
   'Chains': 'A=728-786'}],
 'A1A519': [],
 'A1L190': [{'Source': 'PDB',
   'ID': '6H86',
   'Method': 'X-ray',
   'Resolution': '1.90 A',
   'Chains': 'A/B=1-88'}],
 'A1L3X0': [{'Source': 'PDB',
   'ID': '6Y7F',
   'Method': 'X-ray',
   'Resolution': '2.05 A',
   'Chains': 'A/B=1-281'}],
 'A1X283': [],
 'A2A2Y4': [],
 'A2RU14': [],
 'A2RUB6': [],
 'A2RUC4': [{'Source': 'PDB',
   'ID': '3AL5',
   'Method': 'X-ray',
   'Resolution': '2.50 A',
   'Chains': 'A/B/C/D=1-315'},
  {'Source': 'PDB',
   'ID': '3AL6',
   'Method': 'X-ray',
   'Resolution': '2.80 A',
   'Chains': 'A/B/C/D=1-315'}],
 'A4D1B5': [],
 'A4GXA9': [{'Source': 'PDB',
   'ID': '7F6L',
   'Method': 'X-ray',
   'Resolution': '3.20 A',
   'Chains': 'B=1-379'}],
 'A5D8V7': [{'Source': 'PDB',
   'ID': '8J07',
   'Method': 'EM',
   'Resolution': '4.10 A',
   

In [ ]:
model_count

{'A0A0C5B5G6': 0,
 'A0A1B0GTW7': 0,
 'A0JNW5': 0,
 'A0JP26': 0,
 'A0PK11': 0,
 'A1A4S6': 1,
 'A1A519': 0,
 'A1L190': 1,
 'A1L3X0': 1,
 'A1X283': 0,
 'A2A2Y4': 0,
 'A2RU14': 0,
 'A2RUB6': 0,
 'A2RUC4': 2,
 'A4D1B5': 0,
 'A4GXA9': 1,
 'A5D8V7': 1,
 'A5PLL7': 0,
 'A6BM72': 0,
 'A6H8Y1': 4,
 'A6NCS4': 0,
 'A6NFY7': 0,
 'A6NGG8': 1,
 'A6NI61': 0,
 'A6NKB5': 0,
 'A6NNB3': 0,
 'A7E2V4': 0,
 'A7MCY6': 0,
 'A7MD48': 0,
 'A7XYQ1': 0,
 'A8MQ03': 0,
 'A8MW99': 0,
 'A9UHW6': 0,
 'B1AK53': 0,
 'B2RUY7': 0,
 'B3KU38': 0,
 'B6A8C7': 0,
 'B7U540': 0,
 'C9JLW8': 0,
 'C9JRZ8': 0,
 'D3W0D1': 1,
 'E0CX11': 0,
 'O00115': 0,
 'O00116': 0,
 'O00159': 1,
 'O00161': 2,
 'O00165': 0,
 'O00168': 1,
 'O00214': 36,
 'O00237': 0,
 'O00254': 0,
 'O00268': 31,
 'O00291': 3,
 'O00300': 1,
 'O00322': 0,
 'O00329': 15,
 'O00330': 5,
 'O00337': 0,
 'O00400': 0,
 'O00409': 2,
 'O00422': 1,
 'O00444': 16,
 'O00453': 0,
 'O00462': 0,
 'O00478': 5,
 'O00487': 52,
 'O00505': 0,
 'O00506': 4,
 'O00507': 0,
 'O00560': 55,
 'O005

### Process PDB datas to get
: Chain Name, Chain Position, Valid Chain, Coverage, CovPer
 - Chain Name : ['A/B', 'C/D']
 - Chain Position : ['1-155', '184-317']
 - Valid Chain : ['1-155', '184-317'] (=Chain Position - Transig)
 - Coverage : 289 (=Total length of Valid Chain)
 - CovPer : 91.2 (=Coverage/Total_length*100)

In [ ]:
for key in models: #model[uniprot_id] = [{'Source','ID','Method','Resolution','Chains'}, {"}, ...]
    transig_pos_ = transig_pos[key] #'' or '1-xx'
    transig_end = int(transig_pos_.split('-')[1]) if transig_pos_ else 0 #transit_end = 0 or xx
    valid_length_ = valid_length[key]

    if models[key]: # models = [{},{},...] (else, models = [])
        for model in models[key]: # model = {'Source','ID','Method','Resolution','Chains'} of each key
            Chains = model['Chains'].split(', ') # Chains = ['~=x-x', '~=y-y']
            Chain_names = []
            Chain_pos = []
            Chain_valids = []
            total_coverage = 0

            for chain in Chains:
                Chain = chain.split('=') #Chain = ['~', 'x-x'] or ['-'](NO DATA)
                chain_valid=''
                coverage = 0

                if len(Chain) == 2:
                    chain_name, chain_pos = Chain #chain_pos = 'x-x'
                    try:
                        chain_start, chain_end = map(int, chain_pos.split('-'))
                        if chain_start > transig_end:
                            chain_valid = f'{chain_start}-{chain_end}'
                            coverage = chain_end - chain_start + 1
                        elif chain_end > transig_end:
                            chain_valid = f'{transig_end+1}-{chain_end}'
                            coverage = chain_end - transig_end
                    except: #chain_valid = ''
                        pass

                else: #Chain = ['-']
                    chain_name, chain_pos = 'NO DATA', 'NO DATA'

                total_coverage += coverage

                Chain_valids.append(chain_valid)
                Chain_names.append(chain_name)
                Chain_pos.append(chain_pos)

            CovPer = round(total_coverage / valid_length_ * 100, 1) if valid_length_ != 'error' else 0

            model['Chain Name'] = Chain_names
            model['Chain Position'] = Chain_pos
            model['Valid Chain'] = Chain_valids
            model['Coverage'] = total_coverage
            model['CovPer'] = CovPer

# models
# == [{'Source:''','ID:''','Method:''','Resolution:''','Chains:''','Chain Name:[]','Chain Position:[]','Valid Chain:[]','Coverage':--,'CovPer':--}, {}, {}, {}, ...]

In [ ]:
x = list(models.values())[:100]
y = list(models.keys())[:100]
for i in range(len(x)):
    print(y[i],":", x[i])

A0A0C5B5G6 : []
A0A1B0GTW7 : []
A0JNW5 : []
A0JP26 : []
A0PK11 : []
A1A4S6 : [{'Source': 'PDB', 'ID': '2MIO', 'Method': 'NMR', 'Resolution': '-', 'Chains': 'A=728-786', 'Chain Name': ['A'], 'Chain Position': ['728-786'], 'Valid Chain': ['728-786'], 'Coverage': 59, 'CovPer': 7.5}]
A1A519 : []
A1L190 : [{'Source': 'PDB', 'ID': '6H86', 'Method': 'X-ray', 'Resolution': '1.90 A', 'Chains': 'A/B=1-88', 'Chain Name': ['A/B'], 'Chain Position': ['1-88'], 'Valid Chain': ['1-88'], 'Coverage': 88, 'CovPer': 100.0}]
A1L3X0 : [{'Source': 'PDB', 'ID': '6Y7F', 'Method': 'X-ray', 'Resolution': '2.05 A', 'Chains': 'A/B=1-281', 'Chain Name': ['A/B'], 'Chain Position': ['1-281'], 'Valid Chain': ['1-281'], 'Coverage': 281, 'CovPer': 100.0}]
A1X283 : []
A2A2Y4 : []
A2RU14 : []
A2RUB6 : []
A2RUC4 : [{'Source': 'PDB', 'ID': '3AL5', 'Method': 'X-ray', 'Resolution': '2.50 A', 'Chains': 'A/B/C/D=1-315', 'Chain Name': ['A/B/C/D'], 'Chain Position': ['1-315'], 'Valid Chain': ['1-315'], 'Coverage': 315, 'CovPer': 

### Filter models (with Resolution, CovPer)

In [ ]:
resol_cut, covper_cut = 2.50, 80 #set cutoff values

In [ ]:
filtered_models = {}
filtered_counts = {}

for key in models:
  valid_length_ = valid_length[key]
  filtered_model= []
  filtered_count = 0
  if models[key]:
    for model in models[key]:
      resol = float(model['Resolution'].split(' ')[0]) if model['Resolution'] != '-' else 10
      covper = model['CovPer']
      if resol <= resol_cut and covper >= covper_cut:
        filtered_model.append(model['ID'])
        filtered_count += 1
  filtered_models[key] = filtered_model
  filtered_counts[key] = filtered_count

In [ ]:
list(filtered_models.values())[:100]
list(filtered_counts.values())[:100]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 4,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 2,
 0,
 0,
 7,
 0,
 0,
 0,
 5,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 37,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

## Result : Datas into DataFrame

In [40]:
df = pd.DataFrame({'Protein name' : protein_names,
                   'Type' : transig_type,
                   'Position' : transig_pos,
                   'Total Length' : total_length,
                   'Valid length' : valid_length,
                   'Valid position' : valid_pos,
                   'PDB count' : model_count,
                   'PDB models' : models,
                   'Filtered count' : filtered_counts,
                   'Filtered ID (2.50A & 80%)' : filtered_models})

In [41]:
df.head(20)

,Protein name,Type,Position,Total Length,Valid length,Valid position,PDB count,PDB models,Filtered count,Filtered ID (2.50A & 80%)
A0A0C5B5G6,Mitochondrial-derived peptide MOTS-c (Mitochon...,,,16,16,1-16,0,[],0,[]
A0A1B0GTW7,Ciliated left-right organizer metallopeptidase...,SIGNAL,1-20,788,768,21-788,0,[],0,[]
A0JNW5,Bridge-like lipid transfer protein family memb...,,,1464,1464,1-1464,0,[],0,[]
A0JP26,POTE ankyrin domain family member B3,,,581,581,1-581,0,[],0,[]
A0PK11,Clarin-2,,,232,232,1-232,0,[],0,[]
A1A4S6,Rho GTPase-activating protein 10 (GTPase regul...,,,786,786,1-786,1,"[{'Source': 'PDB', 'ID': '2MIO', 'Method': 'NM...",0,[]
A1A519,Protein FAM170A (Zinc finger domain-containing...,,,330,330,1-330,0,[],0,[]
A1L190,Synaptonemal complex central element protein 3...,,,88,88,1-88,1,"[{'Source': 'PDB', 'ID': '6H86', 'Method': 'X-...",1,[6H86]
A1L3X0,Very long chain fatty acid elongase 7 (EC 2.3....,,,281,281,1-281,1,"[{'Source': 'PDB', 'ID': '6Y7F', 'Method': 'X-...",1,[6Y7F]
A1X283,SH3 and PX domain-containing protein 2B (Adapt...,,,911,911,1-911,0,[],0,[]


## Only 2,260 IDs have filtered model (resol < 2.50 A & CovPer > 80%)
 - Total : 20,417 IDs
 - count_count : distribution of model_count {0: 11906, 1: 2063, 2: 1213, ...}
 => 11,906 IDs have 0 PDB model
 => 8,511 IDs have PDB model

 - filtered_count_count : distribution of filtered_count {0: 18157, 1: 810, 2: 385, 3: 203, 4: 134,...} => 18,157 IDs have 0 filtered_PDB model

In [ ]:
count_count = {} #distribution of model_count {0: 11906, 1: 2063, 2: 1213, ...} => 11906 IDs have 0 PDB model
sorted_count = {} #sorted by size (==count_count)

for num in model_count.values():
    if num in count_count:
        count_count[num] += 1
    else:
        count_count[num] = 1

for k in sorted(count_count):
    sorted_count[k]=count_count[k]

sorted_count #{0: 11906, 1: 2063, 2: 1213, ..., 1116: 1, 1196: 1}

{0: 11906,
 1: 2063,
 2: 1213,
 3: 830,
 4: 709,
 5: 455,
 6: 362,
 7: 286,
 8: 223,
 9: 228,
 10: 174,
 11: 179,
 12: 112,
 13: 119,
 14: 109,
 15: 81,
 16: 67,
 17: 56,
 18: 55,
 19: 63,
 20: 49,
 21: 45,
 22: 49,
 23: 46,
 24: 37,
 25: 42,
 26: 39,
 27: 28,
 28: 18,
 29: 28,
 30: 14,
 31: 20,
 32: 23,
 33: 22,
 34: 14,
 35: 14,
 36: 17,
 37: 15,
 38: 11,
 39: 14,
 40: 17,
 41: 13,
 42: 11,
 43: 9,
 44: 11,
 45: 9,
 46: 9,
 47: 9,
 48: 11,
 49: 39,
 50: 9,
 51: 8,
 52: 16,
 53: 6,
 54: 9,
 55: 9,
 56: 5,
 57: 3,
 58: 8,
 59: 4,
 60: 6,
 61: 8,
 62: 7,
 63: 3,
 64: 12,
 65: 8,
 66: 7,
 67: 17,
 68: 23,
 69: 9,
 70: 5,
 71: 1,
 72: 1,
 73: 5,
 74: 5,
 75: 2,
 76: 1,
 77: 3,
 78: 4,
 79: 3,
 81: 1,
 82: 2,
 83: 3,
 84: 5,
 85: 1,
 86: 2,
 87: 3,
 88: 5,
 89: 3,
 90: 3,
 91: 4,
 92: 3,
 93: 5,
 94: 6,
 95: 8,
 96: 2,
 97: 5,
 98: 2,
 99: 4,
 100: 4,
 101: 6,
 102: 6,
 103: 11,
 104: 3,
 105: 5,
 106: 6,
 107: 3,
 108: 7,
 109: 4,
 110: 3,
 111: 5,
 112: 3,
 113: 4,
 114: 6,
 115: 1,
 116

In [ ]:
filtered_count_count = {} #distribution of filtered_count {0: 18157, 1: 810, 2: 385, 3: 203, 4: 134,...} => 18157 IDs have 0 filtered_PDB model
filtered_sorted_count = {} #sorted by size (==filtered_count_count)

for num in filtered_counts.values():
    if num in filtered_count_count:
        filtered_count_count[num] += 1
    else:
        filtered_count_count[num] = 1

for k in sorted(filtered_count_count):
    filtered_sorted_count[k]=filtered_count_count[k]

filtered_sorted_count #{0: 18157, 1: 810, 2: 385, ... ,876: 1, 1104: 1} in 2.5 A & 80%

{0: 18157,
 1: 810,
 2: 385,
 3: 203,
 4: 134,
 5: 123,
 6: 64,
 7: 59,
 8: 36,
 9: 37,
 10: 23,
 11: 21,
 12: 52,
 13: 17,
 14: 29,
 15: 20,
 16: 17,
 17: 35,
 18: 11,
 19: 9,
 20: 10,
 21: 9,
 22: 5,
 23: 6,
 24: 6,
 25: 3,
 26: 3,
 27: 3,
 28: 6,
 29: 3,
 30: 4,
 31: 3,
 32: 4,
 33: 3,
 34: 8,
 35: 3,
 36: 2,
 37: 4,
 38: 2,
 39: 4,
 40: 4,
 41: 1,
 42: 1,
 44: 2,
 45: 1,
 46: 1,
 47: 1,
 48: 6,
 49: 5,
 50: 1,
 52: 1,
 53: 3,
 55: 1,
 57: 2,
 60: 1,
 61: 1,
 62: 1,
 63: 2,
 64: 1,
 66: 2,
 68: 1,
 69: 2,
 70: 1,
 71: 1,
 73: 3,
 77: 2,
 80: 2,
 82: 1,
 84: 1,
 87: 1,
 88: 1,
 90: 2,
 92: 1,
 97: 1,
 102: 1,
 105: 1,
 106: 1,
 134: 1,
 137: 1,
 138: 1,
 151: 1,
 152: 1,
 174: 1,
 194: 1,
 205: 1,
 210: 1,
 217: 1,
 218: 1,
 240: 1,
 260: 1,
 262: 1,
 265: 1,
 275: 1,
 287: 1,
 380: 1,
 381: 1,
 394: 1,
 432: 1,
 876: 1,
 1104: 1}

In [83]:
https://rest.uniprot.org/uniprotkb/search?query=accession:Q9HBH1&fields=xref_alphafolddb&format=tsv

SyntaxError: invalid syntax (<ipython-input-83-bfefca94410e>, line 1)